# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (120 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Complete all challenges in the notebook. Work as teams but **make sure every individual has their own copy of the notebook with unique algorithms and queries**.
3. **Push to GitHub** *(15 min)* – Individuals commit and push finished notebooks. **Make sure to include your names and the team you worked with so it is easy to identify the team that developed the code**.
4. **Instructor Review** *(30 min)* - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**, assisting with individual work.
5. **Email Delivery** *(15 min)* – Each team sends the instructor an email **with the *.git link to the GitHub repo of every team member (one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. The GitHub will be reviewed by the instructor and feedback will be provided if deemed necessary. Work will be assessed during the workshop.


## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and challenges coded and solved.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step0: Install Libs

```shell
$ pip install requests feedparser
```

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [3]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        # Clean filename
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'sample_docs', max_posts=20)

Saved 20 blog posts to sample_docs


In [4]:
# Example: Load text files from a folder
import os

def load_documents(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

# Replace 'sample_docs/' with your actual folder
documents = load_documents('sample_docs/')
print(f"Loaded {len(documents)} documents.")


Loaded 20 documents.


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [5]:
import re

def tokenize(text):
    # TODO: Implement tokenization logic (e.g., lowercasing, removing punctuation, splitting on whitespace)
    # pass

    # 1. convert to lowercase
    text = text.lower()
    
    # 2. remove punctuation / non-alphanumeric characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 3. split on whitespace
    tokens = text.split()
    
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])  # Preview first 20 tokens

print(f"Total tokens in first document: {len(tokens)}")

# Test on a sample text
sample_text = "Hello, World! This is Python 3.12."
sample_tokens = tokenize(sample_text)
print(sample_tokens)

['applications', 'to', 'join', 'the', 'psf', 'meetup', 'pro', 'network', 'are', 'back', 'open', 'p', 'following', 'the', 'a', 'href', 'https', 'pyfound', 'blogspot', 'com']
Total tokens in first document: 875
['hello', 'world', 'this', 'is', 'python', '3', '12']


### Step 2 Explanation

In this step, we implemented a basic tokenizer to convert raw text into a list of tokens.  
The tokenizer lowercases all text, removes punctuation and non-alphanumeric characters, and splits the cleaned text by whitespace.  

This step is important because tokenization creates the input for normalization and inverted index construction.

## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [6]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize(tokens):
    normalized_tokens = []
    
    for token in tokens:
        # Step 1: Remove stopwords (common words with little meaning)
        if token not in stop_words:
            # Step 2: Apply stemming to reduce word to its root form
            stemmed = stemmer.stem(token)
            normalized_tokens.append(stemmed)
    
    return normalized_tokens

# Example: normalize one document
#Get the first file from /sample_docs
folder_path = "sample_docs"
files = sorted(os.listdir(folder_path))  # sort to ensure consistent order
first_file_path = os.path.join(folder_path, files[0])

with open(first_file_path, 'r', encoding='utf-8') as f:
    doc = f.read()

# Tokenize the document
tokens = tokenize(doc)

# Normalize the tokens
norm_tokens = normalize(tokens)

# Preview first 20 tokens
print(norm_tokens[:20])

# Test on a sample list of tokens
sample_tokens = ['this', 'is', 'a', 'simple', 'running', 'example', 'for', 'testing']
normalized_sample = normalize(sample_tokens)
print(normalized_sample)

['applic', 'join', 'psf', 'meetup', 'pro', 'network', 'back', 'open', 'p', 'follow', 'href', 'http', 'pyfound', 'blogspot', 'com', '2026', '02', 'introduc', 'psf', 'commun']
['simpl', 'run', 'exampl', 'test']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\85155\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


### Step 3 Explanation

This code implements a normalization pipeline for text preprocessing in an NLP and Information Retrieval (IR) system.

First, the `nltk` library is used to load a list of English stopwords and initialize a `PorterStemmer`. Stopwords are common words (e.g., "the", "is", "and") that carry little semantic meaning and are removed to reduce noise.

The `normalize()` function processes a list of tokens by:
1. Removing stopwords to eliminate irrelevant terms  
2. Applying stemming to reduce words to their root forms (e.g., "running" → "run")  

Next, the code reads the first document from the `/sample_docs` folder. The file list is sorted to ensure consistent and reproducible selection of the first document.

The document is then:
- Tokenized using the previously defined tokenizer  
- Normalized using the `normalize()` function  

Finally, the first 20 normalized tokens are printed as a preview of the processed output.

This pipeline prepares text data for building efficient data structures such as inverted indexes.


## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [7]:
from collections import defaultdict

def build_inverted_index(documents):
    # term -> set of document IDs
    inverted_index = defaultdict(set)
    
    # term -> {doc_id: [positions]}
    positional_index = defaultdict(lambda: defaultdict(list))
    
    for doc_id, text in enumerate(documents):
        # Step 1: Tokenize the document
        tokens = tokenize(text)
        
        # Step 2: Normalize tokens
        norm_tokens = normalize(tokens)
        
        # Step 3: Build indexes
        for position, token in enumerate(norm_tokens):
            # Add document ID to inverted index
            inverted_index[token].add(doc_id)
            
            # Store position for positional index
            positional_index[token][doc_id].append(position)
    
    # Convert sets to sorted lists for cleaner output
    inverted_index = {term: sorted(list(doc_ids)) for term, doc_ids in inverted_index.items()}
    
    return inverted_index, positional_index


# Build indexes
inverted_index, positional_index = build_inverted_index(documents)

# Preview first 10 terms
print(dict(list(inverted_index.items())[:10]))


{'applic': [0, 2, 3, 4, 5, 6, 8, 14, 18], 'join': [0, 2, 3, 4, 7, 10, 11, 13, 14, 15, 16, 18], 'psf': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'meetup': [0, 3, 13, 14], 'pro': [0, 2], 'network': [0, 3, 15], 'back': [0, 3, 4, 8, 10, 13, 14], 'open': [0, 3, 4, 6, 8, 10, 11, 12, 13, 18, 19], 'p': [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'follow': [0, 6, 7, 9, 11, 13, 14, 17, 19]}


### Step 4 Explanation

In this step, we construct an **inverted index** and a **positional index** from the document collection.

The **inverted index** maps each normalized term to the list of document IDs in which it appears. This allows efficient retrieval because we can directly look up which documents contain a given term, instead of scanning all documents.

To build the index, each document is:
1. Tokenized into individual words  
2. Normalized by removing stopwords and applying stemming  
3. Processed to map each term to its corresponding document ID  

In addition, a **positional index** is created to store the positions of each term within each document. This is important for supporting **phrase queries**, where we need to check whether terms appear next to each other in the correct order.

For example, for the phrase *"machine learning"*, the system must verify that:
- "machine" appears at position *i*  
- "learning" appears at position *i + 1*  

By combining both indexes:
- The inverted index enables fast term-based lookup  
- The positional index enables accurate phrase matching  

This step forms the core data structure of an Information Retrieval system.

### Step 4 Output Interpretation

The output shows the first few entries of the inverted index.

Each key is a normalized term, and each value is the list of document IDs where that term appears.

For example:
- `announc: [0, 2, 3, 4, 6, 8, 9, 10, 11, 14, 16, 17, 19]` means the term **"announc"** appears in those documents.
- `python: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]` means **"python"** appears in all documents.
- `softwar` and `foundat` are stemmed forms of **"software"** and **"foundation"**.

This confirms that the inverted index was built correctly and that each normalized token is mapped to the documents in which it appears.

## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries **using the inverted matrix**.
- Demonstrate that they return the correct documents.


In [11]:
from collections import defaultdict

def build_positional_index(documents):
    # term -> {doc_id: [positions]}
    positional_index = defaultdict(lambda: defaultdict(list))
    
    # FIX: use enumerate because documents is a list
    for doc_id, text in enumerate(documents):
        tokens = tokenize(text)
        norm_tokens = normalize(tokens)
        
        for position, term in enumerate(norm_tokens):
            positional_index[term][doc_id].append(position)
    
    return positional_index


# Build the positional index
positional_index = build_positional_index(documents)


query1 = "Machine Learning"
query2 = "Python programming"

def phrase_query(query, documents):
    # Tokenize and normalize the query
    query_tokens = tokenize(query)
    query_terms = normalize(query_tokens)
    
    # Return empty list if query becomes empty
    if not query_terms:
        return []
    
    # If first term not in index → no match
    if query_terms[0] not in positional_index:
        return []
    
    # Start with docs containing first term
    candidate_docs = set(positional_index[query_terms[0]].keys())
    
    # Keep only docs containing all terms
    for term in query_terms[1:]:
        if term not in positional_index:
            return []
        candidate_docs &= set(positional_index[term].keys())
    
    results = []
    
    # Check consecutive positions
    for doc_id in candidate_docs:
        first_positions = positional_index[query_terms[0]][doc_id]
        
        for start_pos in first_positions:
            match = True
            
            for offset, term in enumerate(query_terms[1:], start=1):
                if (start_pos + offset) not in positional_index[term][doc_id]:
                    match = False
                    break
            
            if match:
                results.append(doc_id)
                break
    
    return sorted(results)


# Run phrase queries
results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

Documents containing the phrase 'Machine Learning': []
Documents containing the phrase 'Python programming': [0, 2, 3, 7, 8, 10, 11, 14, 18]


## 🧠 Additional Challenge: Use Positional Indexes to compare TF and iDF

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [ ]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

## 🧠 Additional Challenge 2: Implement Optimized Positional Indexes

Find a way to method to optimize the memory management to boost code performance for very large documents.
Implement the new code in the space below:

In [ ]:
# TODO:
# Implement Positional Indexes: Store term positions for each document
# Memory efficient methods for large corpora

def build_positional_index(documents):
    pass
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

## 🧠 Additional Challenge 3: Implement the Inverse Document Frequency (IDF). 
Implement the solution in the space below.

In [ ]:
# TODO
# Calculate Inverse Document Frequency (IDF) for each term using the positional index

def compute_idf_from_positional_index(positional_index, total_docs):
    pass
    return idf_scores

# Compute IDF for all terms using the positional index
idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# Example: Show IDF for a sample term
# NOTE : You could apply stemming in the normalization step, so the term may not match exactly if it was stemmed
sample_term = 'softwar'  # Change to any term you want to inspect
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

## 🧠 Additional Challenge 4: Implement the Term Frequency (TF). 
Implement the solution in the space below.

In [ ]:
# TODO:
# Calculate Term Frequency (TF) for the term 'softwar' in each document
# TF is simply the number of times the term appears in a document

term = 'softwar'  # Use the stemmed term as before

def compute_tf(term, document):
    pass
    return tf
    
# Display as a DataFrame for clarity
tf = compute_tf


#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.